# 01 — UTC Clock Correction (Aeris Ultra321 & Pico017)

The Aeris Ultra321 and Pico017 internal clocks may be misconfigured relative to UTC by large
amounts (seconds to hours). The RPi (WYO deployments) and Toughbook (MML deployments) co-logger
files contain **both** the logger's system clock epoch and the instrument's timestamp in the same
row, making the per-row offset directly computable.

**Goal — coarse alignment, not absolute UTC accuracy.**
The purpose of this step is to remove gross Aeris clock errors (e.g. −60 s, −84 s, +6 hr) so that
the Aeris timestamps are in roughly the same reference frame as the Picarro, which carries reliable
UTC. A residual uncertainty of a few seconds is expected and acceptable — the precise
instrument-to-instrument lag is resolved by cross-correlation in Stage 3 (`03_instrument_aligned/`).
This step just needs to be close enough that Stage 3 is looking in the right neighborhood.

**The logger clock is the best available reference, not guaranteed true UTC.**
The RPi and Toughbook are not GPS- or NTP-synced during field deployments — they free-run from
their last sync before leaving the lab. The offset we compute is therefore the Aeris clock error
*relative to the logger clock*, not relative to a certified UTC source. Both clocks drift, and we
cannot determine which is drifting relative to true UTC without an external reference. For the
purpose of aligning with the Picarro this is sufficient.

**All three Aeris file types share the same internal clock.**
Each Aeris measurement session produces three files stamped with identical internal-clock values:
- **Raw** — gas concentrations and basic instrument state (~20 cols, 1-line header + CSV)
- **Eng** — full engineering diagnostics including GPS, temperatures, fit windows, power (~60 cols, same format)
- **Spectra / Spectralite** — per-measurement spectra (~1,034 cols, headerless)

The same bulk offset is applied to all three types for a given file, preserving their pairing
exactly throughout the rest of the pipeline. WYO_aerisultra460 has trusted timestamps and is
not processed here — its Raw, Eng, and Spectralite files go directly to Stage 02 from `raw/`.

**Key insight**: The offset is NOT globally constant across the campaign — it varies by deployment
period (different clock states, instrument resets). We compute the offset per logger file in Part 2,
then in Part 3 each Aeris file is matched to the logger entry whose UTC window contains that file's
corrected first timestamp.

**Pipeline position:** `raw/` → **`01_utc_corrected/`** → `02_standardized/` → `03_instrument_aligned/`

## Workflow
- **Part 1**: Verify the offset on a clean file pair (Ultra321 Feb 2, RPi — large stable −84 s offset)
- **Part 2**: Survey offset consistency across all logger files — confirms per-deployment variation
- **Part 3**: Match each Aeris file to its logger entry and apply its specific offset to Raw, Eng, and Spectra

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import shutil
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# pipeline/ is one level below the repo root — add root to path for src/ and config
sys.path.insert(0, str(Path('..').resolve()))

from src.aeris_clock import (
    load_logger_file, compute_offset,
    apply_offset_to_raw, apply_offset_to_spectra,
    summarize_logger_files, build_coverage_map,
    batch_assign_offsets, AERIS_TS_FORMAT
)
from paths import RAW_DIR, STAGE_01_DIR, REPO_ROOT
from src.provenance import git_info, check_clean
from datetime import datetime, timezone

OUT_DIR = STAGE_01_DIR
OUT_DIR.mkdir(exist_ok=True)

---
## Part 1 — Single-file verification (Ultra321, Feb 2 WYO)

We use the RPi logger file from Feb 2 (`ultra_20260202_210959.dat`, ~3700 records, 21:09–22:12 UTC)
and a corresponding Aeris raw file from the same deployment (`Ultra100321_260202_204623.txt`).

The Feb 2 session has a large, clean offset of −84 s — the Aeris clock was set roughly 84 seconds
ahead of the logger clock at the start of this deployment. A flat, tight offset line here confirms
the error is stable within the session and a single median is the right correction to apply.
This is what a well-behaved logger file looks like.

In [ ]:
# Load the RPi Ultra logger file from Feb 2
rpi_path = RAW_DIR / 'LANL_rpi/Ultra/ultra_20260202_210959.dat'
rpi_df   = load_logger_file(rpi_path)

t_utc    = pd.to_datetime(rpi_df['Epoch_time'], unit='s', utc=True)
offset_s = compute_offset(rpi_df)

print(f'File          : {rpi_path.name}')
print(f'Records       : {len(rpi_df):,}')
print(f'UTC range     : {t_utc.iloc[0]}  ->  {t_utc.iloc[-1]}')
print(f'Offset median : {offset_s:.3f} s  ({offset_s/3600:.4f} hrs)')
print(f'Offset std    : {rpi_df["offset_s"].std():.4f} s')
rpi_df[['Epoch_time', 'Time Stamp', 'offset_s', 'CH4 (ppm)']].head(5)

In [ ]:
# Plot 1: offset stability + CH4 over the full deployment.
# A flat, tight offset line confirms the clock error is stable within this session —
# a single median value is appropriate to apply. Spread should be sub-second.
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[f'Clock offset: Epoch_time - epoch(Time Stamp)  |  {rpi_path.name}', 'CH4 (ppm)'],
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=t_utc, y=rpi_df['offset_s'], mode='lines',
    line=dict(width=0.8, color='steelblue'), name='offset (s)', showlegend=False
), row=1, col=1)

fig.add_hline(
    y=offset_s, line_dash='dash', line_color='red', line_width=1.5,
    annotation_text=f'median = {offset_s:.1f} s ({offset_s/3600:.4f} hrs)',
    annotation_position='top right', row=1, col=1
)

fig.add_trace(go.Scatter(
    x=t_utc, y=rpi_df['CH4 (ppm)'], mode='lines',
    line=dict(width=0.8, color='black'), name='CH4', showlegend=False
), row=2, col=1)

fig.update_yaxes(title_text='Offset (s)', row=1, col=1)
fig.update_yaxes(title_text='CH4 (ppm)', row=2, col=1)
fig.update_xaxes(title_text='Time (UTC)', row=2, col=1)
fig.update_layout(height=500)
fig.show()

The ~1-minute sawtooth (~4 s amplitude) is a beat-frequency artifact from two clocks running at
slightly different rates. The Aeris outputs data at approximately 1 Hz by its internal clock; the
RPi timestamps each serial line by its own independent clock. If the Aeris is outputting at
slightly less than exactly 1 Hz (e.g. 56 records/min rather than 60), the fractional-second
mismatch accumulates until it resets at the next integer-second boundary, producing a sawtooth
whose period is the time it takes that beat to complete one cycle.


In [ ]:
# Load the corresponding Aeris Ultra321 raw file from the same Feb 2 deployment
aeris_path = RAW_DIR / 'LANL_aerisultra321/Raw/Ultra100321_260202_204623.txt'
aeris_df   = pd.read_csv(aeris_path)
aeris_df.columns = aeris_df.columns.str.strip()

ts_wrong     = pd.to_datetime(aeris_df['Time Stamp'].str.strip(), format=AERIS_TS_FORMAT)
ts_corrected = (ts_wrong + pd.Timedelta(seconds=offset_s)).dt.tz_localize('UTC')

print(f'Aeris file    : {aeris_path.name}')
print(f'Records       : {len(aeris_df):,}')
print(f'Wrong range   : {ts_wrong.iloc[0]}  ->  {ts_wrong.iloc[-1]}')
print(f'Corrected UTC : {ts_corrected.iloc[0]}  ->  {ts_corrected.iloc[-1]}')

In [ ]:
# Plot 2: overlay RPi CH4 (correct UTC) and Aeris CH4 (corrected timestamps).
# The two traces should overlap in the window where both files have data.
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t_utc, y=rpi_df['CH4 (ppm)'], mode='lines',
    line=dict(width=2), opacity=0.7,
    name=f'RPi logger — correct UTC ({rpi_path.name})'
))

fig.add_trace(go.Scatter(
    x=ts_corrected, y=aeris_df['CH4 (ppm)'], mode='lines',
    line=dict(width=1.5, dash='dash', color='tomato'), opacity=0.9,
    name=f'Aeris raw — corrected +{offset_s:.1f}s ({aeris_path.name})'
))

fig.update_layout(
    title='Verification: RPi (correct epoch) vs Aeris (corrected timestamps) — should overlap where both have data',
    xaxis_title='Time (UTC)',
    yaxis_title='CH4 (ppm)',
    height=400,
    legend=dict(yanchor='bottom', y=0.01, xanchor='right', x=0.99)
)
fig.show()

---
## Part 2 — Offset survey across all logger files

Compute the offset from every RPi and Toughbook logger file to characterise the full campaign.

Two things to look for:
- **High std** within a file (>1 s) — indicates startup records or a transitional clock state; the median is still usable but flag it.
- **Discrete offset jumps** between files — the expected pattern. The Aeris clock is reset between deployments and each reset lands at a new wrong value. This is why per-deployment matching is needed: a global correction would be wrong by tens of seconds for many files.

The Ultra321 shows several distinct offset groups from different clock resets. The Pico017 has two completely separate eras: a **timezone misconfiguration** (~+6 hr = +21604 s) in Jan–Feb 4, and a corrected era from Feb 5 onward where it slowly drifts at ~+1 s/day. These are shown in separate panels below so neither story is crushed by the other's scale.

In [ ]:
print('=== LANL_rpi / Ultra ===')
rpi_ultra_summary = summarize_logger_files(RAW_DIR / 'LANL_rpi/Ultra')
pd.set_option('display.max_colwidth', 45)
rpi_ultra_summary

In [ ]:
print('=== LANL_rpi / Pico ===')
rpi_pico_summary = summarize_logger_files(RAW_DIR / 'LANL_rpi/Pico')
rpi_pico_summary

In [ ]:
print('=== LANL_toughbook / Ultra ===')
tb_ultra_summary = summarize_logger_files(RAW_DIR / 'LANL_toughbook/Ultra')
tb_ultra_summary

In [ ]:
print('=== LANL_toughbook / Pico ===')
tb_pico_summary = summarize_logger_files(RAW_DIR / 'LANL_toughbook/Pico')
tb_pico_summary

In [ ]:
# Offset survey — 3 panels so each story is legible at its own scale.
#
# Blue  = logger file used in the coverage map (≥200 records) — offset applied to Aeris files
# Grey  = excluded startup artifact (<200 records) — session too short to trust, skipped
#
# Error bars removed from the plot. Hover each point to see the within-file std.
# A large std on a blue point flags a file where a few bad records inflated the spread;
# the median offset is still reliable but worth inspecting.
all_ultra = pd.concat([rpi_ultra_summary, tb_ultra_summary], ignore_index=True)
all_pico  = pd.concat([rpi_pico_summary,  tb_pico_summary],  ignore_index=True)

pico_tz  = all_pico[all_pico['offset_median_s'] >  1000]  # wrong-timezone era (~+6 hr)
pico_ok  = all_pico[all_pico['offset_median_s'] <= 1000]  # corrected era

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=[
        'Ultra321 — discrete clock resets between deployments',
        'Pico017 — Jan 19 to Feb 4: timezone misconfiguration (~+6 hr)',
        'Pico017 — Feb 5 onward: corrected era, slow drift (~+1 s/day)',
    ],
    vertical_spacing=0.15
)

for row, df in [(1, all_ultra), (2, pico_tz), (3, pico_ok)]:
    s = df.dropna(subset=['offset_median_s', 'start_utc'])
    for mask, color, name in [
        (s['n_records'] >= 200, '#1f77b4', 'Used in correction (≥200 records)'),
        (s['n_records'] <  200, '#aaaaaa', 'Excluded — startup artifact (<200 records)'),
    ]:
        sub = s[mask]
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=sub['start_utc'],
            y=sub['offset_median_s'],
            mode='markers',
            marker=dict(size=9, color=color),
            customdata=sub[['filename', 'n_records', 'offset_std_s']],
            hovertemplate=(
                '<b>%{customdata[0]}</b><br>'
                'offset = %{y:.2f} s<br>'
                'std    = %{customdata[2]:.2f} s<br>'
                'n      = %{customdata[1]:,}<br>'
                '%{x|%Y-%m-%d %H:%M UTC}<extra></extra>'
            ),
            name=name,
            legendgroup=name,
            showlegend=(row == 1),
        ), row=row, col=1)
    fig.update_yaxes(title_text='Offset (s)', row=row, col=1)
    fig.update_xaxes(title_text='Logger file start (UTC)', row=row, col=1)

fig.update_layout(
    height=950,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

In [ ]:
# Summary of offset groups — confirms per-deployment matching is the right strategy.
# Within-file std should be <1 s for clean files; flag anything higher before applying.
print('Ultra321 — offset by deployment group:')
for _, r in all_ultra.dropna(subset=['offset_median_s']).sort_values('start_utc').iterrows():
    flag = '  ← FLAG: high std' if r['offset_std_s'] > 1.0 and r['n_records'] >= 200 else ''
    excl = '  (excluded — startup)' if r['n_records'] < 200 else ''
    print(f"  {r['filename']}  offset={r['offset_median_s']:+.1f}s  std={r['offset_std_s']:.2f}s  n={int(r['n_records']):,}{flag}{excl}")

print()
print('Pico017 — wrong-timezone era (Jan 19 – Feb 4):')
for _, r in pico_tz.dropna(subset=['offset_median_s']).sort_values('start_utc').iterrows():
    excl = '  (excluded — startup)' if r['n_records'] < 200 else ''
    print(f"  {r['filename']}  offset={r['offset_median_s']:+.0f}s (~{r['offset_median_s']/3600:.1f} hr)  std={r['offset_std_s']:.3f}s  n={int(r['n_records']):,}{excl}")

print()
print('Pico017 — corrected era (Feb 5+):')
for _, r in pico_ok.dropna(subset=['offset_median_s']).sort_values('start_utc').iterrows():
    excl = '  (excluded — startup)' if r['n_records'] < 200 else ''
    print(f"  {r['filename']}  offset={r['offset_median_s']:+.2f}s  std={r['offset_std_s']:.3f}s  n={int(r['n_records']):,}{excl}")

**Flagged files — both medians are still reliable.**

`ultra_20260203_162046.dat` — std = 4.26 s, n = 26,890, offset = −60.1 s

This is the main Feb 3 deployment session (16:20 → 23:50 UTC). The high std comes from a small
cluster of initialization records at the start of the session where the Aeris clock hadn't yet
settled — even a large logger session can carry a handful of embedded initialization records near
its open, separate from the dedicated startup files that were excluded. The median is robust to
a few outliers in ~27 k records. The offset jumps from −84 s (Feb 2) to −60 s here because the
instrument was power-cycled between deployments and the clock reset to a new wrong value. To
inspect this directly, load the file with `load_logger_file` and plot `offset_s` vs time — you
will see the outlier cluster at the start and a flat line at −60 s for the rest of the session.

`ultra_20260208_000001.dat` — std = 1.99 s, n = 86,723, offset = +2.3 s

This is an auto-split midnight file (logger writes a new file at 00:00:01). The slightly elevated
std is likely due to a few records right at the midnight boundary where the logger briefly lagged
before the new file opened cleanly. The median (+2.3 s) is consistent with the surrounding
drift trend (Feb 7: +1.4 s, Feb 9: +3.3 s) and is the correct value to apply.

---
## Part 3 — Batch: apply per-deployment offset to all Raw, Eng, and Spectra files

Offsets vary across the campaign (Part 2 confirmed this). Each Aeris file is matched to the logger
entry whose corrected UTC window contains that file's first timestamp. `batch_assign_offsets` tries
each logger entry's `offset_median_s` and checks whether the corrected timestamp falls within that
logger's UTC window (±2 h buffer). Raw and Eng files use `file_type='raw'` (1-line header + CSV);
Spectra use `file_type='spectra'` (headerless, timestamp in col 0).

**Step 1** — Build coverage maps and preview all assignments: check for any unmatched files (`status != 'ok'`).  
**Steps 2–7** — Write corrected files. Run each independently; all are safe to re-run.  
**Step 8** — Save `ts_offsets.json` manifest. **Must be run after Steps 2–7.**

Output mirrors the `raw/` directory structure under `01_utc_corrected/`, adding `Eng/`:

    01_utc_corrected/
      LANL_aerisultra321/
        Raw/
          <corrected files...>
          no_coverage/         ← unmatched files copied as-is (timestamps uncorrected)
        Eng/
          <corrected files...>
          no_coverage/
        Spectra/
          <corrected files...>
          no_coverage/
      LANL_aerispico017/
        Raw/   (+ no_coverage/)
        Eng/   (+ no_coverage/)
        Spectra/  (+ no_coverage/)
      ts_offsets.json          ← matched assignments + no_coverage inventory (Step 8)

In [ ]:
# Build per-instrument coverage maps from Part 2 summaries.
# Files with < 200 records (startup artifacts) and errored files are excluded automatically.
# Duplicate filenames across RPi and Toughbook dirs are deduplicated (first occurrence wins).
ultra_coverage = build_coverage_map(rpi_ultra_summary, tb_ultra_summary)
pico_coverage  = build_coverage_map(rpi_pico_summary,  tb_pico_summary)

print(f'Ultra coverage entries : {len(ultra_coverage)}')
for e in ultra_coverage:
    print(f"  [{e['start_utc']}  ->  {e['end_utc']}]  offset={e['offset_s']:+.1f}s  ({e['logger_filename']})")

print()
print(f'Pico  coverage entries : {len(pico_coverage)}')
for e in pico_coverage:
    print(f"  [{e['start_utc']}  ->  {e['end_utc']}]  offset={e['offset_s']:+.1f}s  ({e['logger_filename']})")

In [ ]:
# Step 1 — Assign offsets and preview results. Nothing is written here.
# Defines ultra_raw_files, ultra_eng_files, ultra_raw_assign, etc. — required by write cells below.
ultra_raw_files     = sorted((RAW_DIR / 'LANL_aerisultra321/Raw').glob('*.txt'))
ultra_eng_files     = sorted((RAW_DIR / 'LANL_aerisultra321/Eng').glob('*.txt'))
ultra_spectra_files = sorted((RAW_DIR / 'LANL_aerisultra321/Spectra').glob('*.txt'))
pico_raw_files      = sorted((RAW_DIR / 'LANL_aerispico017/Raw').glob('*.txt'))
pico_eng_files      = sorted((RAW_DIR / 'LANL_aerispico017/Eng').glob('*.txt'))
pico_spectra_files  = sorted((RAW_DIR / 'LANL_aerispico017/Spectra').glob('*.txt'))

ultra_raw_assign     = batch_assign_offsets(ultra_raw_files,     ultra_coverage, file_type='raw')
ultra_eng_assign     = batch_assign_offsets(ultra_eng_files,     ultra_coverage, file_type='raw')
ultra_spectra_assign = batch_assign_offsets(ultra_spectra_files, ultra_coverage, file_type='spectra')
pico_raw_assign      = batch_assign_offsets(pico_raw_files,      pico_coverage,  file_type='raw')
pico_eng_assign      = batch_assign_offsets(pico_eng_files,      pico_coverage,  file_type='raw')
pico_spectra_assign  = batch_assign_offsets(pico_spectra_files,  pico_coverage,  file_type='spectra')

def _line_count(path):
    with open(path, 'rb') as f:
        return sum(1 for _ in f)

for label, df, src_dir in [
    ('Ultra321 Raw',     ultra_raw_assign,     RAW_DIR / 'LANL_aerisultra321/Raw'),
    ('Ultra321 Eng',     ultra_eng_assign,     RAW_DIR / 'LANL_aerisultra321/Eng'),
    ('Ultra321 Spectra', ultra_spectra_assign,  RAW_DIR / 'LANL_aerisultra321/Spectra'),
    ('Pico017 Raw',      pico_raw_assign,       RAW_DIR / 'LANL_aerispico017/Raw'),
    ('Pico017 Eng',      pico_eng_assign,       RAW_DIR / 'LANL_aerispico017/Eng'),
    ('Pico017 Spectra',  pico_spectra_assign,   RAW_DIR / 'LANL_aerispico017/Spectra'),
]:
    n_ok   = (df['status'] == 'ok').sum()
    n_fail = (df['status'] != 'ok').sum()
    status = 'ALL MATCHED' if n_fail == 0 else f'{n_fail} UNMATCHED — review before applying'
    print(f'{label:20s}: {n_ok:3d} matched  |  {status}')
    if n_fail:
        bad = df[df['status'] != 'ok'].copy()
        bad['n_lines'] = bad['filename'].apply(lambda fn: _line_count(src_dir / fn))
        print(bad[['filename', 'n_lines', 'status']].to_string(index=False))
    print()

For this deployment, all of the above look reasonable -- most of the January and early February data (before Feb 03) was having the instruments run in my office without the logger on. The timestamps don't really matter but the raw data may be useful to characterize concentration drifts. 

In [ ]:
# ── WRITES FILES TO DISK ──────────────────────────────────────────────────────
# For each matched file (status == 'ok'):
#   reads the raw CSV from RAW_DIR, shifts "Time Stamp" by offset_s, writes to OUT_DIR
# For each unmatched file (status != 'ok'):
#   no logger coverage found — copied unchanged to no_coverage/ for reference
# Safe to re-run: existing files are overwritten.

out_raw_dir = OUT_DIR / 'LANL_aerisultra321/Raw'
no_cov_dir  = out_raw_dir / 'no_coverage'
n = len(ultra_raw_assign)

print(f'Ultra321 Raw  ({len(ultra_raw_files)} files)')
print(f'  src : {RAW_DIR / "LANL_aerisultra321/Raw"}')
print(f'  dest: {out_raw_dir}\n')

for i, (_, row) in enumerate(ultra_raw_assign.iterrows()):
    prog    = f'({i+1}/{n})'
    in_path = RAW_DIR / 'LANL_aerisultra321/Raw' / row['filename']
    if row['status'] != 'ok':
        no_cov_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(in_path, no_cov_dir / row['filename'])
        print(f'  {prog}  NO_COV  {row["filename"]}  ({row["status"]})')
        continue
    out_path = out_raw_dir / row['filename']
    apply_offset_to_raw(in_path, row['offset_s'], out_path)
    print(f'  {prog}  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')

print('\nDone.')

In [ ]:
# ── WRITES FILES TO DISK ──────────────────────────────────────────────────────
# Ultra321 Eng files — same offset logic as Raw (same format: 1-line header + CSV).

out_eng_dir = OUT_DIR / 'LANL_aerisultra321/Eng'
no_cov_dir  = out_eng_dir / 'no_coverage'
n = len(ultra_eng_assign)

print(f'Ultra321 Eng  ({len(ultra_eng_files)} files)')
print(f'  src : {RAW_DIR / "LANL_aerisultra321/Eng"}')
print(f'  dest: {out_eng_dir}\n')

for i, (_, row) in enumerate(ultra_eng_assign.iterrows()):
    prog    = f'({i+1}/{n})'
    in_path = RAW_DIR / 'LANL_aerisultra321/Eng' / row['filename']
    if row['status'] != 'ok':
        no_cov_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(in_path, no_cov_dir / row['filename'])
        print(f'  {prog}  NO_COV  {row["filename"]}  ({row["status"]})')
        continue
    out_path = out_eng_dir / row['filename']
    apply_offset_to_raw(in_path, row['offset_s'], out_path)
    print(f'  {prog}  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')

print('\nDone.')

In [ ]:
# ── WRITES FILES TO DISK ──────────────────────────────────────────────────────
# Same logic as the Raw cell above, but for Spectra files.
# Spectra are headerless CSVs — apply_offset_to_spectra shifts column 0 (the timestamp).
# Spectra files are large; expect several minutes total.

out_spectra_dir = OUT_DIR / 'LANL_aerisultra321/Spectra'
no_cov_dir      = out_spectra_dir / 'no_coverage'
n = len(ultra_spectra_assign)

print(f'Ultra321 Spectra  ({len(ultra_spectra_files)} files)')
print(f'  src : {RAW_DIR / "LANL_aerisultra321/Spectra"}')
print(f'  dest: {out_spectra_dir}\n')

for i, (_, row) in enumerate(ultra_spectra_assign.iterrows()):
    prog    = f'({i+1}/{n})'
    in_path = RAW_DIR / 'LANL_aerisultra321/Spectra' / row['filename']
    if row['status'] != 'ok':
        no_cov_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(in_path, no_cov_dir / row['filename'])
        print(f'  {prog}  NO_COV  {row["filename"]}  ({row["status"]})')
        continue
    out_path = out_spectra_dir / row['filename']
    apply_offset_to_spectra(in_path, row['offset_s'], out_path)
    print(f'  {prog}  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')

print('\nDone.')

In [ ]:
# ── WRITES FILES TO DISK ──────────────────────────────────────────────────────
# Same logic as Ultra321 Raw above, but for Pico017 Raw files.

out_raw_dir = OUT_DIR / 'LANL_aerispico017/Raw'
no_cov_dir  = out_raw_dir / 'no_coverage'
n = len(pico_raw_assign)

print(f'Pico017 Raw  ({len(pico_raw_files)} files)')
print(f'  src : {RAW_DIR / "LANL_aerispico017/Raw"}')
print(f'  dest: {out_raw_dir}\n')

for i, (_, row) in enumerate(pico_raw_assign.iterrows()):
    prog    = f'({i+1}/{n})'
    in_path = RAW_DIR / 'LANL_aerispico017/Raw' / row['filename']
    if row['status'] != 'ok':
        no_cov_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(in_path, no_cov_dir / row['filename'])
        print(f'  {prog}  NO_COV  {row["filename"]}  ({row["status"]})')
        continue
    out_path = out_raw_dir / row['filename']
    apply_offset_to_raw(in_path, row['offset_s'], out_path)
    print(f'  {prog}  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')

print('\nDone.')

In [ ]:
# ── WRITES FILES TO DISK ──────────────────────────────────────────────────────
# Pico017 Eng files — same offset logic as Raw.

out_eng_dir = OUT_DIR / 'LANL_aerispico017/Eng'
no_cov_dir  = out_eng_dir / 'no_coverage'
n = len(pico_eng_assign)

print(f'Pico017 Eng  ({len(pico_eng_files)} files)')
print(f'  src : {RAW_DIR / "LANL_aerispico017/Eng"}')
print(f'  dest: {out_eng_dir}\n')

for i, (_, row) in enumerate(pico_eng_assign.iterrows()):
    prog    = f'({i+1}/{n})'
    in_path = RAW_DIR / 'LANL_aerispico017/Eng' / row['filename']
    if row['status'] != 'ok':
        no_cov_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(in_path, no_cov_dir / row['filename'])
        print(f'  {prog}  NO_COV  {row["filename"]}  ({row["status"]})')
        continue
    out_path = out_eng_dir / row['filename']
    apply_offset_to_raw(in_path, row['offset_s'], out_path)
    print(f'  {prog}  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')

print('\nDone.')

In [ ]:
# ── WRITES FILES TO DISK ──────────────────────────────────────────────────────
# Same logic as Ultra321 Spectra above, but for Pico017 Spectra files.
# Spectra files are large; expect several minutes total.

out_spectra_dir = OUT_DIR / 'LANL_aerispico017/Spectra'
no_cov_dir      = out_spectra_dir / 'no_coverage'
n = len(pico_spectra_assign)

print(f'Pico017 Spectra  ({len(pico_spectra_files)} files)')
print(f'  src : {RAW_DIR / "LANL_aerispico017/Spectra"}')
print(f'  dest: {out_spectra_dir}\n')

for i, (_, row) in enumerate(pico_spectra_assign.iterrows()):
    prog    = f'({i+1}/{n})'
    in_path = RAW_DIR / 'LANL_aerispico017/Spectra' / row['filename']
    if row['status'] != 'ok':
        no_cov_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(in_path, no_cov_dir / row['filename'])
        print(f'  {prog}  NO_COV  {row["filename"]}  ({row["status"]})')
        continue
    out_path = out_spectra_dir / row['filename']
    apply_offset_to_spectra(in_path, row['offset_s'], out_path)
    print(f'  {prog}  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')

print('\nDone.')

In [ ]:
# ── STEP 8: SAVE MANIFEST — run after all six write cells above ───────────────
# Records every matched assignment (offset_s + logger source) and every unmatched
# file (line count + reason) for all six file types. Written to 01_utc_corrected/.

def _to_records(assign_df):
    return [
        {'filename': r['filename'], 'offset_s': r['offset_s'], 'logger': r['logger_filename']}
        for _, r in assign_df[assign_df['status'] == 'ok'].iterrows()
    ]

def _to_no_coverage(assign_df, src_dir):
    records = []
    for _, r in assign_df[assign_df['status'] != 'ok'].iterrows():
        p = src_dir / r['filename']
        with open(p, 'rb') as f:
            n_lines = sum(1 for _ in f)
        records.append({'filename': r['filename'], 'n_lines': n_lines, 'reason': r['status']})
    return records

offset_record = {
    'ultra321_raw':     _to_records(ultra_raw_assign),
    'ultra321_eng':     _to_records(ultra_eng_assign),
    'ultra321_spectra': _to_records(ultra_spectra_assign),
    'pico017_raw':      _to_records(pico_raw_assign),
    'pico017_eng':      _to_records(pico_eng_assign),
    'pico017_spectra':  _to_records(pico_spectra_assign),
    'no_coverage': {
        'ultra321_raw':     _to_no_coverage(ultra_raw_assign,     RAW_DIR / 'LANL_aerisultra321/Raw'),
        'ultra321_eng':     _to_no_coverage(ultra_eng_assign,     RAW_DIR / 'LANL_aerisultra321/Eng'),
        'ultra321_spectra': _to_no_coverage(ultra_spectra_assign, RAW_DIR / 'LANL_aerisultra321/Spectra'),
        'pico017_raw':      _to_no_coverage(pico_raw_assign,      RAW_DIR / 'LANL_aerispico017/Raw'),
        'pico017_eng':      _to_no_coverage(pico_eng_assign,      RAW_DIR / 'LANL_aerispico017/Eng'),
        'pico017_spectra':  _to_no_coverage(pico_spectra_assign,  RAW_DIR / 'LANL_aerispico017/Spectra'),
    }
}
check_clean(REPO_ROOT, context='Stage 01')
git_hash, git_dirty = git_info(REPO_ROOT)
offset_record['stage']      = '01_utc_correction'
offset_record['run_utc']    = datetime.now(timezone.utc).isoformat()
offset_record['git_hash']   = git_hash
offset_record['git_dirty']  = git_dirty

out_json = OUT_DIR / 'ts_offsets.json'
with open(out_json, 'w') as f:
    json.dump(offset_record, f, indent=2)

total_matched = sum(len(offset_record[k]) for k in
    ('ultra321_raw', 'ultra321_eng', 'ultra321_spectra',
     'pico017_raw',  'pico017_eng',  'pico017_spectra'))
total_no_cov  = sum(len(v) for v in offset_record['no_coverage'].values())
print(f'Saved: {out_json}')
print(f'  {total_matched} matched assignments, {total_no_cov} no_coverage files recorded')